# LeetCode 2018 — Check if Word Can Be Placed In Crossword bằng Python

Link đề: https://leetcode.com/problems/check-if-word-can-be-placed-in-crossword/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def placeWordInCrossword(self, board: List[List[str]], word: str) -> bool:
        ...
```

## 1. Mục tiêu học

1. Dịch chính xác 4 điều kiện của đề, đặc biệt điều kiện "không có ô trống/chữ cái **ngay trước và ngay sau** từ" ⇒ từ phải lấp **trọn** một **đoạn tối đa** giữa các ô `#` (hoặc mép bảng).
2. Duyệt lưới theo **4 hướng** bằng vector hướng `(dr, dc)` và kiểm tra biên.
3. Quy bài toán về **hàng và cột**: tách mỗi hàng/cột thành các đoạn theo `#`, rồi so khớp với `word` **và** `word[::-1]` (thay cho hai chiều ngược).
4. Dùng các công cụ Python gọn: `zip(*board)` để lấy cột, `"".join(...).split("#")` để tách đoạn.
5. Kiểm thử đối chiếu cách "thử mọi vị trí" với cách "theo đoạn" trên bảng ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Cho bảng ô chữ `board` kích thước `m × n`, mỗi ô là chữ thường (đã điền), `' '` (trống), hoặc `'#'` (bị chặn). Một từ `word` đặt được **ngang** (trái→phải hoặc phải→trái) hoặc **dọc** (trên→dưới hoặc dưới→trên) nếu:

1. Không chiếm ô `'#'`.
2. Mỗi chữ cái nằm ở ô trống `' '` hoặc ô đã có **đúng** chữ cái đó.
3. Nếu đặt ngang: **ngay bên trái và ngay bên phải** từ không được là ô trống hay chữ cái (tức là phải là `'#'` hoặc ngoài bảng).
4. Nếu đặt dọc: **ngay phía trên và ngay phía dưới** từ cũng vậy.

Trả về `True` nếu đặt được, ngược lại `False`.

**Ví dụ 1:** `board = [["#"," ","#"],[" "," ","#"],["#","c"," "]]`, `word = "abc"` → `True` (cột 1, trên→dưới).

**Ví dụ 2:** `board = [[" ","#","a"],[" ","#","c"],[" ","#","a"]]`, `word = "ac"` → `False` (cột nào cũng dài 3, luôn có ô thừa phía trên/dưới).

**Ví dụ 3:** `board = [["#"," ","#"],[" "," ","#"],["#"," ","c"]]`, `word = "ca"` → `True` (hàng 2, phải→trái: `c` ở cột 2, `a` ở cột 1).

**Ràng buộc:** `1 ≤ m·n ≤ 2·10⁵`, `1 ≤ len(word) ≤ max(m, n)`.

In [1]:
import random
import time
from typing import List

examples = [
    ([["#", " ", "#"], [" ", " ", "#"], ["#", "c", " "]], "abc", True),
    ([[" ", "#", "a"], [" ", "#", "c"], [" ", "#", "a"]], "ac", False),
    ([["#", " ", "#"], [" ", " ", "#"], ["#", " ", "c"]], "ca", True),
]
def show_board(board):
    for row in board:
        print("|" + "|".join(ch if ch != " " else "_" for ch in row) + "|")

for b, w, exp in examples:
    show_board(b); print(f"word={w!r} -> kỳ vọng {exp}"); print()

|#|_|#|
|_|_|#|
|#|c|_|
word='abc' -> kỳ vọng True

|_|#|a|
|_|#|c|
|_|#|a|
word='ac' -> kỳ vọng False

|#|_|#|
|_|_|#|
|#|_|c|
word='ca' -> kỳ vọng True



## 3. Kiến thức nền

### 3.1 Vector hướng và kiểm tra biên

Bốn hướng: phải `(0,1)`, trái `(0,−1)`, xuống `(1,0)`, lên `(−1,0)`. Ô thứ `t` của từ bắt đầu ở `(r, c)` theo hướng `(dr, dc)` là `(r + t·dr, c + t·dc)`. Ô "ngay trước" là `t = −1`, "ngay sau" là `t = k` (với `k = len(word)`).

In [2]:
DIRS = [(0, 1), (0, -1), (1, 0), (-1, 0)]
m, n = 3, 3
def inside(r, c):
    return 0 <= r < m and 0 <= c < n
r, c, (dr, dc), k = 0, 1, (1, 0), 3
print("các ô của từ:", [(r + t * dr, c + t * dc) for t in range(k)])
print("ô ngay trước:", (r - dr, c - dc), "trong bảng?", inside(r - dr, c - dc))
print("ô ngay sau:", (r + k * dr, c + k * dc), "trong bảng?", inside(r + k * dr, c + k * dc))

các ô của từ: [(0, 1), (1, 1), (2, 1)]
ô ngay trước: (-1, 1) trong bảng? False
ô ngay sau: (3, 1) trong bảng? False


### 3.2 `zip(*board)` — chuyển vị để lấy các cột

`zip(*board)` gom phần tử cùng chỉ số của mỗi hàng ⇒ các **cột**. Nhờ đó "đặt dọc" trở thành "đặt ngang" trên bảng chuyển vị.

In [3]:
board = [["#", " ", "#"], [" ", " ", "#"], ["#", "c", " "]]
cols = [list(col) for col in zip(*board)]
print("các cột:", cols)

các cột: [['#', ' ', '#'], [' ', ' ', 'c'], ['#', '#', ' ']]


### 3.3 Tách đoạn bằng `split("#")`

`"".join(row).split("#")` tách hàng thành các **đoạn tối đa** không chứa `#`. Các đoạn rỗng (`""`) xuất hiện khi có hai `#` liền nhau hoặc `#` ở mép — bỏ qua được vì `len(word) ≥ 1`.

In [4]:
row = ["a", " ", "#", "#", " ", "b", " ", "#"]
print("".join(row).split("#"))

['a ', '', ' b ', '']


### 3.4 Đảo chuỗi `[::-1]` thay cho chiều ngược

Đặt `word` từ phải sang trái trên đoạn `seg` ⇔ đặt `word[::-1]` từ trái sang phải. Vậy chỉ cần kiểm **một chiều** với hai chuỗi `word` và `word[::-1]`.

In [5]:
def fits(seg, w):
    return len(seg) == len(w) and all(s == " " or s == ch for s, ch in zip(seg, w))

seg = " c"     # hàng 2 của ví dụ 3, bỏ '#'
print(fits(seg, "ca"), fits(seg, "ca"[::-1]))

False True


## 4. Ý tưởng nền tảng / trực giác

Điều kiện 3–4 nói: ô ngay trước và ngay sau từ phải là `#` hoặc ngoài bảng. Kết hợp điều kiện 1 (không chiếm `#`): từ phải **lấp đầy đúng một đoạn tối đa** giữa hai "vách" (`#` hoặc mép).

Xét ví dụ 2: các đoạn dọc là cột 0 `"   "` (dài 3), cột 2 `"aca"` (dài 3); đoạn ngang: `" "`, `"a"`, `" "`, `"c"`, `" "`, `"a"` (dài 1). Không có đoạn nào dài **đúng 2** ⇒ `False`, dù `"ac"` xuất hiện trong cột 2.

Ví dụ 1: cột 1 là `"  c"` (dài 3) — khớp `"abc"` (ô trống nhận `a`, `b`; ô thứ ba đã là `c`) ⇒ `True`.

In [6]:
def all_segments(board):
    segs = []
    for i, row in enumerate(board):
        for s in "".join(row).split("#"):
            if s: segs.append(("hàng", i, s))
    for j, col in enumerate(zip(*board)):
        for s in "".join(col).split("#"):
            if s: segs.append(("cột", j, s))
    return segs

for b, w, _ in examples[:2]:
    print([f"{kind} {idx}: {s!r}" for kind, idx, s in all_segments(b)], "| word =", w)

["hàng 0: ' '", "hàng 1: '  '", "hàng 2: 'c '", "cột 0: ' '", "cột 1: '  c'", "cột 2: ' '"] | word = abc
["hàng 0: ' '", "hàng 0: 'a'", "hàng 1: ' '", "hàng 1: 'c'", "hàng 2: ' '", "hàng 2: 'a'", "cột 0: '   '", "cột 2: 'aca'"] | word = ac


## 5. Cách 1 — Brute Force / baseline: thử mọi ô bắt đầu × 4 hướng

**Ý tưởng:** làm đúng 4 điều kiện của đề cho mọi `(r, c, hướng)`:
- ô trước (`t = −1`) và ô sau (`t = k`) phải ngoài bảng hoặc là `#`;
- mọi ô `t = 0..k−1` phải trong bảng, khác `#`, và là `' '` hoặc bằng `word[t]`.

In [7]:
def can_place_brute(board, word):
    m, n, k = len(board), len(board[0]), len(word)
    def blocked(r, c):
        return not (0 <= r < m and 0 <= c < n) or board[r][c] == "#"
    for r in range(m):
        for c in range(n):
            for dr, dc in DIRS:
                if not blocked(r - dr, c - dc):
                    continue
                if not blocked(r + k * dr, c + k * dc):
                    continue
                ok = True
                for t in range(k):
                    rr, cc = r + t * dr, c + t * dc
                    if not (0 <= rr < m and 0 <= cc < n) or board[rr][cc] not in (" ", word[t]):
                        ok = False; break
                if ok:
                    return True
    return False

for b, w, exp in examples:
    got = can_place_brute(b, w); print(got, "kỳ vọng", exp); assert got == exp

True kỳ vọng True
False kỳ vọng False
True kỳ vọng True


Lưu ý `board[rr][cc] not in (" ", word[t])` cũng loại `#` (vì `#` không phải chữ thường).

**Độ phức tạp:** `O(m·n·4·k)`; với `k` tới `max(m, n)` có thể ~`2·10⁵ · 4 · 450` (bảng vuông) hoặc tệ hơn với bảng dài một hàng (`k` tới `2·10⁵`, nhưng khi đó điều kiện biên loại sớm hầu hết vị trí). Space `O(1)`.

**Nút thắt:** nhiều vị trí bắt đầu bị loại ngay bởi điều kiện "ô trước phải bị chặn" — tức là chỉ **đầu các đoạn** mới có thể là điểm bắt đầu. Ta nên duyệt thẳng các đoạn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** một vị trí hợp lệ ⇔ một **đoạn tối đa** (trong một hàng hoặc một cột) có `len == k` và khớp `word` theo một trong hai chiều.

**Quy tắc "nếu… thì…":**
- **Nếu** đoạn có độ dài khác `k` **thì** bỏ qua (điều kiện biên sẽ vi phạm hoặc từ không vừa).
- **Nếu** độ dài bằng `k` **thì** kiểm `fits(seg, word)` hoặc `fits(seg, word[::-1])`.
- Mỗi ô của đoạn khớp khi là `' '` hoặc trùng chữ cái.

**Invariant khi quét một hàng:** biến `start` luôn là chỉ số đầu của đoạn đang mở (ngay sau `#` gần nhất hoặc 0). Khi gặp `#` hoặc hết hàng, đoạn `[start, j)` đóng lại và được kiểm tra.

Mỗi ô thuộc đúng một đoạn ngang và một đoạn dọc ⇒ tổng độ dài các đoạn là `2·m·n` ⇒ `O(m·n)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quét từng hàng/cột, tự cắt đoạn bằng chỉ số

**Giải thích từng đoạn:**
1. Hàm `check_line(line)`: duyệt `j` từ `0..len(line)`; khi `j == len` hoặc `line[j] == '#'` thì đoạn `line[start:j]` đóng.
2. Nếu đoạn dài `k` và khớp `word` hoặc `rev` ⇒ `True`.
3. Áp dụng cho mọi hàng và mọi cột.

In [8]:
def can_place_lines(board, word):
    k, rev = len(word), word[::-1]
    def match(line, lo, w):
        return all(line[lo + t] == " " or line[lo + t] == w[t] for t in range(k))
    def check_line(line):
        start = 0
        for j in range(len(line) + 1):
            if j == len(line) or line[j] == "#":
                if j - start == k and (match(line, start, word) or match(line, start, rev)):
                    return True
                start = j + 1
        return False
    return any(check_line(row) for row in board) or any(check_line(col) for col in zip(*board))

for b, w, exp in examples:
    assert can_place_lines(b, w) == exp
print([can_place_lines(b, w) for b, w, _ in examples])

[True, False, True]


In [9]:
class SolutionLines:
    def placeWordInCrossword(self, board: List[List[str]], word: str) -> bool:
        k, rev = len(word), word[::-1]
        def match(line, lo, w):
            for t in range(k):
                ch = line[lo + t]
                if ch != " " and ch != w[t]:
                    return False
            return True
        def check_line(line):
            start = 0
            for j in range(len(line) + 1):
                if j == len(line) or line[j] == "#":
                    if j - start == k and (match(line, start, word) or match(line, start, rev)):
                        return True
                    start = j + 1
            return False
        return any(check_line(r) for r in board) or any(check_line(c) for c in zip(*board))

print([SolutionLines().placeWordInCrossword(b, w) for b, w, _ in examples])

[True, False, True]


**Độ phức tạp Cách 2:** mỗi ô được duyệt một lần trong hàng và một lần trong cột; mỗi đoạn dài `k` được so khớp tối đa 2 lần `O(k)` — tổng `O(m·n)`. Space `O(m·n)` cho `zip(*board)` (có thể tránh bằng chỉ số).

### 7.2 Cách 3 — `split("#")` gọn gàng (bản nộp chính)

Dựng chuỗi cho mỗi hàng và cột, tách theo `#`, so khớp từng đoạn.

In [10]:
def can_place_split(board, word):
    k, rev = len(word), word[::-1]
    def fits(seg, w):
        return all(s == " " or s == ch for s, ch in zip(seg, w))
    lines = ["".join(row) for row in board] + ["".join(col) for col in zip(*board)]
    for line in lines:
        for seg in line.split("#"):
            if len(seg) == k and (fits(seg, word) or fits(seg, rev)):
                return True
    return False

for b, w, exp in examples:
    assert can_place_split(b, w) == exp
print([can_place_split(b, w) for b, w, _ in examples])

[True, False, True]


**Độ phức tạp Cách 3:** `O(m·n)` thời gian và bộ nhớ (các chuỗi tạm). Ngắn gọn, ít lỗi chỉ số.

### 7.3 Cách 4 — Thử mọi vị trí nhưng chỉ ở **đầu đoạn**, 2 hướng cho mỗi trục

Một bước trung gian giữa Cách 1 và Cách 2: chỉ thử các ô `(r, c)` mà ô bên trái (hoặc phía trên) bị chặn — tức là đầu đoạn ngang (hoặc dọc) — rồi kiểm cả `word` và `rev` theo chiều phải (hoặc xuống). Cách này giữ nguyên tinh thần "mô phỏng đặt từ" nhưng tránh thử 4 hướng ở mọi ô.

In [11]:
def can_place_heads(board, word):
    m, n, k, rev = len(board), len(board[0]), len(word), word[::-1]
    def blocked(r, c):
        return not (0 <= r < m and 0 <= c < n) or board[r][c] == "#"
    def try_place(r, c, dr, dc, w):
        for t in range(k):
            rr, cc = r + t * dr, c + t * dc
            if blocked(rr, cc) or board[rr][cc] not in (" ", w[t]):
                return False
        return blocked(r + k * dr, c + k * dc)
    for r in range(m):
        for c in range(n):
            if board[r][c] == "#":
                continue
            if blocked(r, c - 1) and (try_place(r, c, 0, 1, word) or try_place(r, c, 0, 1, rev)):
                return True
            if blocked(r - 1, c) and (try_place(r, c, 1, 0, word) or try_place(r, c, 1, 0, rev)):
                return True
    return False

for b, w, exp in examples:
    assert can_place_heads(b, w) == exp
print([can_place_heads(b, w) for b, w, _ in examples])

[True, False, True]


**Độ phức tạp Cách 4:** số đầu đoạn `≤ 2·m·n`, mỗi lần thử dừng sớm khi gặp ô chặn ⇒ tổng `O(m·n)` (mỗi ô thuộc một đoạn ngang và một đoạn dọc, mỗi đoạn được thử 2 lần). Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 (`word = "abc"`, `rev = "cba"`, `k = 3`):

| Dòng | Chuỗi | Các đoạn | Đoạn dài 3? | Khớp? |
|---|---|---|---|---|
| hàng 0 | `"# #"` | `""`, `" "`, `""` | không | — |
| hàng 1 | `"  #"` | `"  "`, `""` | không | — |
| hàng 2 | `"#c "` | `""`, `"c "` | không | — |
| cột 0 | `"# #"` | `""`, `" "`, `""` | không | — |
| cột 1 | `"  c"` | `"  c"` | **có** | `"abc"`: `' '`↔a ✓, `' '`↔b ✓, `c`↔c ✓ ⇒ **True** |

In [12]:
def trace_split(board, word):
    k, rev = len(word), word[::-1]
    names = [f"hàng {i}" for i in range(len(board))] + [f"cột {j}" for j in range(len(board[0]))]
    lines = ["".join(row) for row in board] + ["".join(col) for col in zip(*board)]
    for name, line in zip(names, lines):
        segs = line.split("#")
        print(f"{name}: {line!r} -> đoạn {segs}")
        for seg in segs:
            if len(seg) == k:
                for w in (word, rev):
                    detail = [f"{s!r}~{ch}:{'ok' if s in (' ', ch) else 'X'}" for s, ch in zip(seg, w)]
                    ok = all(s in (" ", ch) for s, ch in zip(seg, w))
                    print(f"    thử {w!r} trên {seg!r}: {detail} -> {ok}")
                    if ok:
                        return True
    return False

print("Đáp án:", trace_split(*examples[0][:2]))

hàng 0: '# #' -> đoạn ['', ' ', '']
hàng 1: '  #' -> đoạn ['  ', '']
hàng 2: '#c ' -> đoạn ['', 'c ']
cột 0: '# #' -> đoạn ['', ' ', '']
cột 1: '  c' -> đoạn ['  c']
    thử 'abc' trên '  c': ["' '~a:ok", "' '~b:ok", "'c'~c:ok"] -> True
Đáp án: True


In [13]:
print("Đáp án:", trace_split(*examples[1][:2]))

hàng 0: ' #a' -> đoạn [' ', 'a']
hàng 1: ' #c' -> đoạn [' ', 'c']
hàng 2: ' #a' -> đoạn [' ', 'a']
cột 0: '   ' -> đoạn ['   ']
cột 1: '###' -> đoạn ['', '', '', '']
cột 2: 'aca' -> đoạn ['aca']
Đáp án: False


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def placeWordInCrossword(self, board: List[List[str]], word: str) -> bool:
        k, rev = len(word), word[::-1]

        def fits(seg: str, w: str) -> bool:
            # mỗi ô phải trống hoặc trùng chữ cái
            return all(s == " " or s == ch for s, ch in zip(seg, w))

        lines = ["".join(row) for row in board]               # các hàng
        lines += ["".join(col) for col in zip(*board)]        # các cột
        for line in lines:
            for seg in line.split("#"):                       # đoạn tối đa giữa các '#'
                if len(seg) == k and (fits(seg, word) or fits(seg, rev)):
                    return True
        return False

for b, w, exp in examples:
    assert Solution().placeWordInCrossword(b, w) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề (đặt hợp lệ ⇔ lấp đầy đoạn tối đa).** Xét đặt ngang từ trái sang phải trên hàng `r`, chiếm cột `c..c+k−1`.
- (⇒) Điều kiện 1 ⇒ các ô `c..c+k−1` đều khác `#`. Điều kiện 3 ⇒ ô `c−1` và `c+k` là `#` hoặc ngoài bảng. Vậy `[c, c+k)` là một **đoạn tối đa** không chứa `#`, có độ dài `k`. Điều kiện 2 ⇒ `fits(seg, word)`.
- (⇐) Nếu một đoạn tối đa `[c, c+k)` có `fits(seg, word)` thì đặt `word` vào đó thỏa cả 4 điều kiện.

Đặt phải→trái ⇔ đặt `word[::-1]` trái→phải trên cùng đoạn. Đặt dọc tương tự trên các cột (bảng chuyển vị). Cách 3 liệt kê **đúng** mọi đoạn tối đa của mọi hàng và cột (`split('#')`) và kiểm cả hai chiều ⇒ trả `True` ⇔ tồn tại cách đặt.

## 11. Phân tích Time / Space

Gọi `m × n` là kích thước bảng, `k = len(word)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi ô × 4 hướng | `O(m·n·k)` worst | `O(1)` |
| 2. Quét đoạn bằng chỉ số | `O(m·n)` | `O(m·n)` (do `zip`) |
| 3. `split('#')` | `O(m·n)` | `O(m·n)` |
| 4. Chỉ đầu đoạn | `O(m·n)` | `O(1)` |

Đo trên bảng lớn `450 × 450` (≈ `2·10⁵` ô), toàn ô trống và vài `#`, từ dài:

In [15]:
rng = random.Random(2018)
N = 450
big = [[" " if rng.random() > 0.02 else "#" for _ in range(N)] for _ in range(N)]
w = "".join(rng.choice("ab") for _ in range(N))       # dài bằng cả hàng -> gần như không đặt được
for name, fn in [("brute", can_place_brute), ("lines", can_place_lines), ("split", can_place_split),
                 ("heads", can_place_heads)]:
    t = time.perf_counter(); r = fn(big, w)
    print(f"{name:6s}: {r} ({time.perf_counter() - t:.3f}s)")

brute : False (0.199s)
lines : False (0.018s)
split : False (0.003s)
heads : False (0.178s)


## 12. Bộ test

Ví dụ đề; biên (bảng 1×1, từ 1 chữ, hàng không có `#`, từ khớp nhưng thừa ô, palindrome, chỉ khớp chiều ngược, chữ cái xung đột); đối chiếu 4 cách trên bảng ngẫu nhiên nhỏ (seed cố định) gồm `' '`, `'#'`, và vài chữ cái.

In [16]:
fns = {
    "brute": can_place_brute, "lines": can_place_lines, "split": can_place_split, "heads": can_place_heads,
    "Solution": lambda b, w: Solution().placeWordInCrossword(b, w),
    "SolutionLines": lambda b, w: SolutionLines().placeWordInCrossword(b, w),
}
edge = [
    ([[" "]], "a", True),
    ([["#"]], "a", False),
    ([["b"]], "a", False),
    ([[" ", " ", " "]], "ab", False),              # đoạn dài 3, từ dài 2
    ([[" ", " ", "#"]], "ab", True),
    ([["b", " ", "#"]], "ab", True),               # chỉ khớp phải->trái
    ([["a", "c", " "]], "abc", False),             # xung đột chữ 'c' ở vị trí 'b'
    ([["#", "a", "#"], [" ", "b", " "], ["#", " ", "#"]], "aba", True),    # cột 1 = "ab " (ô cuối trống nhận 'a')
    ([["#", "a", "#"], [" ", "c", " "], ["#", " ", "#"]], "aba", False),   # cột 1 = "ac ": 'c' xung đột với 'b'
]
for b, w, exp in [(b, w, e) for b, w, e in examples] + edge:
    for name, fn in fns.items():
        assert fn(b, w) == exp, (name, b, w)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [17]:
rng = random.Random(42)
cnt_true = 0
for _ in range(1500):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    b = [[rng.choice([" ", " ", " ", "#", "a", "b"]) for _ in range(n)] for _ in range(m)]
    w = "".join(rng.choice("ab") for _ in range(rng.randint(1, max(m, n))))
    want = can_place_brute(b, w)
    cnt_true += want
    for name, fn in fns.items():
        assert fn(b, w) == want, (name, b, w)
print(f"1500 test ngẫu nhiên ({cnt_true} True): mọi cách khớp brute force")

1500 test ngẫu nhiên (881 True): mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên chiều ngược (phải→trái, dưới→trên)

Bảng dưới đây chỉ đặt được `"ac"` theo chiều phải→trái (chữ `c` có sẵn ở bên trái).

In [18]:
def wrong_one_direction(board, word):
    k = len(word)
    lines = ["".join(r) for r in board] + ["".join(c) for c in zip(*board)]
    return any(len(seg) == k and all(s in (" ", ch) for s, ch in zip(seg, word))   # SAI: không thử word[::-1]
               for line in lines for seg in line.split("#"))

# Ví dụ 3 tình cờ còn đặt được trái->phải ở hàng 1 ("  "), nên dùng bảng tự dựng chỉ khớp chiều ngược:
b, w = [["#", "#", "#"], ["c", " ", "#"], ["#", "#", "#"]], "ac"
print("sai:", wrong_one_direction(b, w), "| đúng:", can_place_brute(b, w))

sai: False | đúng: True


In [19]:
print("sửa:", Solution().placeWordInCrossword(b, w))

sửa: True


### Lỗi 2 — Không kiểm điều kiện "ngay trước / ngay sau" (cho phép đặt giữa đoạn dài hơn)

Chỉ tìm xem `word` có "nhét vừa" một chỗ nào không. Ví dụ 2 sẽ thành `True` vì `"ac"` khớp hai ô đầu của cột 2.

In [20]:
def wrong_no_boundary(board, word):
    k, rev = len(word), word[::-1]
    lines = ["".join(r) for r in board] + ["".join(c) for c in zip(*board)]
    for line in lines:
        for i in range(len(line) - k + 1):                  # SAI: mọi vị trí, không cần chặn hai đầu
            seg = line[i:i + k]
            if "#" not in seg and any(all(s in (" ", ch) for s, ch in zip(seg, x)) for x in (word, rev)):
                return True
    return False

b, w, exp = examples[1]
print("sai:", wrong_no_boundary(b, w), "| đúng:", exp)

sai: True | đúng: False


In [21]:
print("sửa (đoạn tối đa, len == k):", Solution().placeWordInCrossword(b, w))

sửa (đoạn tối đa, len == k): False


### Lỗi 3 — Dùng `len(seg) >= k` thay vì `==`

Cùng bản chất với lỗi 2 nhưng ở dạng `split`: đoạn dài hơn `k` thì luôn có ô thừa trước/sau từ.

In [22]:
def wrong_ge(board, word):
    k, rev = len(word), word[::-1]
    lines = ["".join(r) for r in board] + ["".join(c) for c in zip(*board)]
    return any(len(seg) >= k and any(all(s in (" ", ch) for s, ch in zip(seg, x)) for x in (word, rev))  # SAI
               for line in lines for seg in line.split("#"))

b, w, exp = examples[1]
print("sai:", wrong_ge(b, w), "| đúng:", exp)

sai: True | đúng: False


In [23]:
print("sửa:", can_place_split(b, w))

sửa: False


### Lỗi 4 — Coi ô có chữ cái là "bị chặn" (chỉ cho đặt vào ô trống)

Đề cho phép ô đã có chữ **trùng** với chữ cần đặt. Ví dụ 1 cần dùng lại chữ `c` có sẵn.

In [24]:
def wrong_only_empty(board, word):
    k, rev = len(word), word[::-1]
    lines = ["".join(r) for r in board] + ["".join(c) for c in zip(*board)]
    return any(len(seg) == k and all(s == " " for s in seg)          # SAI: không cho trùng chữ
               for line in lines for seg in line.split("#"))

b, w, exp = examples[0]
print("sai:", wrong_only_empty(b, w), "| đúng:", exp)

sai: False | đúng: True


In [25]:
print("sửa:", Solution().placeWordInCrossword(b, w))

sửa: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Không có ô trống/chữ ngay trước và sau"** ⇒ từ phải lấp **trọn đoạn tối đa** ⇒ tách theo vách (`#`).
- **4 hướng** ⇒ 2 trục × 2 chiều; chiều ngược = đảo chuỗi.
- **Cột** ⇒ `zip(*board)`; chuỗi ⇒ `"".join(...)`; tách ⇒ `split('#')`.
- So khớp có ký tự đại diện (`' '` khớp mọi chữ) ⇒ `all(s == ' ' or s == ch ...)`.
- Viết brute force theo đúng 4 điều kiện để kiểm tra cách tối ưu — rất hữu ích khi đề có nhiều điều kiện biên.

## 15. Biến thể của bài

1. **Đếm số cách đặt** (bài tập 5): đếm mọi cặp (đoạn, chiều) khớp — chú ý palindrome trên ô toàn trống cho cùng một kết quả theo hai chiều.
2. **Đặt một danh sách từ** sao cho lấp kín bảng (giải ô chữ đầy đủ) — quay lui, NP-khó nói chung.
3. **LeetCode 79 — Word Search:** từ đi theo **đường** bất kỳ (kề cạnh), không phải đường thẳng — DFS.
4. **Ký tự đại diện `?`** trong `word` (khớp mọi chữ): sửa `fits` thành `s == ' ' or ch == '?' or s == ch`.

Minh họa biến thể 4 (tự dựng, kiểm bằng brute force):

In [26]:
def can_place_wild(board, word):
    k, rev = len(word), word[::-1]
    fits = lambda seg, w: all(s == " " or ch == "?" or s == ch for s, ch in zip(seg, w))
    lines = ["".join(r) for r in board] + ["".join(c) for c in zip(*board)]
    return any(len(seg) == k and (fits(seg, word) or fits(seg, rev)) for line in lines for seg in line.split("#"))

def can_place_wild_brute(board, word):
    # thay '?' bằng mọi chữ cái có thể (chỉ 'a','b' để nhỏ) rồi dùng brute force gốc
    from itertools import product
    pos = [i for i, ch in enumerate(word) if ch == "?"]
    for fill in product("abc", repeat=len(pos)):
        w = list(word)
        for i, ch in zip(pos, fill): w[i] = ch
        if can_place_brute(board, "".join(w)): return True
    return False

rng = random.Random(4)
for _ in range(400):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    b = [[rng.choice([" ", " ", "#", "a", "b", "c"]) for _ in range(n)] for _ in range(m)]
    w = "".join(rng.choice("ab?") for _ in range(rng.randint(1, max(m, n))))
    assert can_place_wild(b, w) == can_place_wild_brute(b, w)
print("Biến thể ký tự đại diện '?' khớp brute force")

Biến thể ký tự đại diện '?' khớp brute force


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi ô × 4 hướng | làm đúng 4 điều kiện | `O(m·n·k)` | `O(1)` | kiểm tra |
| 2. Quét đoạn bằng chỉ số | `start` + gặp `#` thì đóng | `O(m·n)` | `O(m·n)` | khi muốn tránh tạo chuỗi |
| 3. `split('#')` | tách đoạn, so 2 chiều | `O(m·n)` | `O(m·n)` | **nộp bài**, ngắn gọn |
| 4. Chỉ đầu đoạn | mô phỏng có cắt tỉa | `O(m·n)` | `O(1)` | bộ nhớ hạn chế |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `board = [[" ", " "], [" ", " "]]`, `word = "ab"`.
2. **Tính tay:** `board = [["a", " ", "#"], ["#", "b", " "], [" ", "#", "c"]]`, `word = "ab"` — liệt kê mọi đoạn và kiểm.
3. **Điền TODO:** `maximal_segments(line)` trả về danh sách `(start, end)` của các đoạn tối đa (không chứa `#`) trong một hàng.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `count_placements(board, word)` đếm số cặp `(ô bắt đầu, hướng)` hợp lệ (theo đúng 4 điều kiện của đề). Kiểm với một brute force đếm.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def maximal_segments(line):
    res = []
    # TODO
    return res

In [28]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_place_word(board, word):
    # TODO
    pass

In [29]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def count_placements(board, word):
    # TODO: với mỗi đoạn dài k: +1 nếu khớp word (chiều xuôi), +1 nếu khớp word[::-1] (chiều ngược)
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Không có `#` ⇒ mỗi hàng/cột là một đoạn dài 2. (2) Các ô đều trống. (3) `True`.

**Bài 2.** (1) Hàng 0: `"a "` (dài 2) khớp `"ab"`? `a`↔a ✓, `' '`↔b ✓. (2) Vậy `True` ngay. (3) Tự liệt kê các đoạn còn lại để luyện.

**Bài 3.** (1) `start = 0`. (2) Gặp `#` tại `j`: nếu `j > start` thêm `(start, j)`; `start = j + 1`. (3) Cuối hàng xử lý đoạn cuối.

**Bài 4.** (1) Lấy hàng và cột (`zip(*board)`). (2) `split('#')`. (3) So `len == k` và hai chiều.

**Bài 5.** (1) Đoạn `[s, e)` trên hàng: chiều phải khởi đầu tại `s`, chiều trái khởi đầu tại `e−1`. (2) Hai cặp (ô, hướng) khác nhau dù palindrome. (3) Đếm cả hàng lẫn cột; với `k = 1`, cùng một ô có thể được đếm ở cả 4 hướng.

## 19. Đáp án tham khảo

In [30]:
# Bài 1, 2
assert Solution().placeWordInCrossword([[" ", " "], [" ", " "]], "ab") is True
assert Solution().placeWordInCrossword([["a", " ", "#"], ["#", "b", " "], [" ", "#", "c"]], "ab") is True
print("Bài 1: True | Bài 2: True")
trace_split([["a", " ", "#"], ["#", "b", " "], [" ", "#", "c"]], "ab")

Bài 1: True | Bài 2: True
hàng 0: 'a #' -> đoạn ['a ', '']
    thử 'ab' trên 'a ': ["'a'~a:ok", "' '~b:ok"] -> True


True

In [31]:
# Bài 3, 4
def maximal_segments_ans(line):
    res, start = [], 0
    for j in range(len(line) + 1):
        if j == len(line) or line[j] == "#":
            if j > start: res.append((start, j))
            start = j + 1
    return res

assert maximal_segments_ans(list("a #  b#")) == [(0, 2), (3, 6)]
assert maximal_segments_ans(list("##")) == []
my_place_word_ans = can_place_split
print("Bài 3:", maximal_segments_ans(list(" a##b  #c")), "| Bài 4: dùng lại Cách 3")

Bài 3: [(0, 2), (4, 7), (8, 9)] | Bài 4: dùng lại Cách 3


In [32]:
# Bài 5
def count_placements_ans(board, word):
    k, rev = len(word), word[::-1]
    fits = lambda seg, w: all(s == " " or s == ch for s, ch in zip(seg, w))
    lines = ["".join(r) for r in board] + ["".join(c) for c in zip(*board)]
    return sum(fits(seg, word) + fits(seg, rev) for line in lines for seg in line.split("#") if len(seg) == k)

def count_placements_brute(board, word):
    m, n, k = len(board), len(board[0]), len(word); total = 0
    blocked = lambda r, c: not (0 <= r < m and 0 <= c < n) or board[r][c] == "#"
    for r in range(m):
        for c in range(n):
            for dr, dc in DIRS:
                if blocked(r - dr, c - dc) and blocked(r + k * dr, c + k * dc) and all(
                        not blocked(r + t * dr, c + t * dc) and board[r + t * dr][c + t * dc] in (" ", word[t])
                        for t in range(k)):
                    total += 1
    return total

rng = random.Random(8)
for _ in range(800):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    b = [[rng.choice([" ", " ", "#", "a", "b"]) for _ in range(n)] for _ in range(m)]
    w = "".join(rng.choice("ab") for _ in range(rng.randint(1, max(m, n))))
    assert count_placements_ans(b, w) == count_placements_brute(b, w)
    assert (count_placements_ans(b, w) > 0) == Solution().placeWordInCrossword(b, w)
print("Bài 5: OK; ví dụ 1 ->", count_placements_ans(*examples[0][:2]), "| bảng 2x2 trống, 'ab' ->",
      count_placements_ans([[" ", " "], [" ", " "]], "ab"))

Bài 5: OK; ví dụ 1 -> 1 | bảng 2x2 trống, 'ab' -> 8


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đặt từ vào đoạn tối đa":**

```python
k, rev = len(word), word[::-1]
fits = lambda seg, w: all(s == ' ' or s == ch for s, ch in zip(seg, w))
lines = [''.join(r) for r in board] + [''.join(c) for c in zip(*board)]
return any(len(seg) == k and (fits(seg, word) or fits(seg, rev))
           for line in lines for seg in line.split('#'))
```

In [33]:
def template(board, word):
    k, rev = len(word), word[::-1]
    fits = lambda seg, w: all(s == " " or s == ch for s, ch in zip(seg, w))
    lines = ["".join(r) for r in board] + ["".join(c) for c in zip(*board)]
    return any(len(seg) == k and (fits(seg, word) or fits(seg, rev)) for line in lines for seg in line.split("#"))
print([template(b, w) for b, w, _ in examples])

[True, False, True]


**Checklist:**

- [ ] Xét cả **hàng** và **cột** (`zip(*board)`).
- [ ] Xét cả hai chiều (`word` và `word[::-1]`).
- [ ] Đoạn phải dài **đúng** `k` (đoạn tối đa giữa các `#`/mép).
- [ ] Ô khớp khi là `' '` **hoặc** đúng chữ cái.
- [ ] Không coi ô có chữ trùng là bị chặn.
- [ ] Đã thử: bảng 1×1, từ 1 chữ, chỉ khớp chiều ngược, đoạn dài hơn từ.